This notebook covers advanced pandas operations for transforming and reshaping data:
- Element-wise operations: `map`, `apply`, `transform`
- Grouping operations: `groupby`, `agg`, group-wise `transform` and `apply`
- Reshaping: `melt`, `pivot`, `pivot_table`

In [ ]:
import pandas as pd
import numpy as np

# Sample dataset for demonstrations
np.random.seed(42)
df = pd.DataFrame({
    'student': ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank'],
    'group': ['A', 'A', 'B', 'B', 'C', 'C'],
    'math': [85, 90, 78, 92, 88, 76],
    'physics': [80, 85, 82, 88, 90, 70],
    'chemistry': [75, 88, 80, 95, 85, 72]
})
df

## Element-wise Operations: `map`, `apply`, `transform`

### `map` - Series Only, Element-wise

`map` is used exclusively on **Series** to apply a function, dictionary, or another Series element-by-element.

**Use cases:**
- Replacing values using a dictionary or function
- Simple element-wise transformations on a single column

In [ ]:
# map with a function
df['math'].map(lambda x: x + 5)  # Add 5 to each math grade

In [ ]:
# map with a dictionary - useful for categorical replacements
group_names = {'A': 'Alpha', 'B': 'Beta', 'C': 'Gamma'}
df['group'].map(group_names)

In [ ]:
# map returns NaN for values not in the dictionary
incomplete_map = {'A': 'Alpha', 'B': 'Beta'}  # Missing 'C'
df['group'].map(incomplete_map)  # 'C' becomes NaN

> **Note:** For dictionary mapping where you want to keep original values for missing keys, use `replace()` instead of `map()`.

In [ ]:
# replace keeps original values for missing keys
df['group'].replace({'A': 'Alpha', 'B': 'Beta'})  # 'C' stays as 'C'

### `apply` - Flexible, Works on Series and DataFrames

`apply` is more versatile than `map`:
- On a **Series**: applies a function element-wise (similar to `map`)
- On a **DataFrame**: applies a function along an axis (rows or columns)

In [ ]:
# apply on Series - similar to map
df['math'].apply(lambda x: x + 5)

In [ ]:
# apply on DataFrame - operates on entire columns (axis=0, default)
df[['math', 'physics', 'chemistry']].apply(lambda col: col.max() - col.min())

In [ ]:
# apply on DataFrame - operates on entire rows (axis=1)
df[['math', 'physics', 'chemistry']].apply(lambda row: row.mean(), axis=1)

**Key insight:** When using `apply` on a DataFrame, the function receives an entire Series (a column or a row), not individual values.

In [ ]:
# Demonstrating what apply receives
def show_input(x):
    print(f"Type: {type(x).__name__}, Shape: {x.shape}")
    print(x)
    print("---")
    return x.sum()

df[['math', 'physics']].apply(show_input)  # Each column is passed as a Series

### `transform` - Same Shape Output, Group-Aware

`transform` always returns a result with the **same shape** as the input. This is crucial for:
- Broadcasting group-level calculations back to individual rows
- Ensuring alignment with the original DataFrame

In [ ]:
# transform on a Series - element-wise, same as map/apply
df['math'].transform(lambda x: x + 5)

In [ ]:
# transform on DataFrame columns
df[['math', 'physics', 'chemistry']].transform(lambda x: (x - x.mean()) / x.std())

### Comparison: When to Use Each

| Method | Works On | Input to Function | Output Shape | Best For |
|--------|----------|-------------------|--------------|----------|
| `map` | Series only | Single value | Same as input | Value replacement, simple transforms |
| `apply` | Series & DataFrame | Value (Series) or Series (DataFrame) | Can differ | Flexible operations, aggregations |
| `transform` | Series & DataFrame | Series | **Must match input** | Normalization, group broadcasting |

In [ ]:
# Equivalent operations - all add 5 to math grades
print("map:      ", df['math'].map(lambda x: x + 5).tolist())
print("apply:    ", df['math'].apply(lambda x: x + 5).tolist())
print("transform:", df['math'].transform(lambda x: x + 5).tolist())

In [ ]:
# Only map works with dictionaries directly
df['group'].map({'A': 'Alpha', 'B': 'Beta', 'C': 'Gamma'})
# df['group'].apply({'A': 'Alpha'})  # Would raise an error
# df['group'].transform({'A': 'Alpha'})  # Would raise an error

In [ ]:
# Only apply can reduce dimensions (return different shape)
df[['math', 'physics', 'chemistry']].apply(lambda x: x.sum())  # Returns a Series with 3 values
# df[['math', 'physics', 'chemistry']].transform(lambda x: x.sum())  # Would broadcast the sum to all rows

## Grouping Operations

### `groupby` - Split-Apply-Combine

The `groupby` operation follows a **split-apply-combine** pattern:
1. **Split** the data into groups based on one or more keys
2. **Apply** a function to each group independently
3. **Combine** the results into a new data structure

In [ ]:
# Creating a groupby object
grouped = df.groupby('group')
print(type(grouped))
print(f"Number of groups: {grouped.ngroups}")
print(f"Groups: {grouped.groups}")

In [ ]:
# Iterating over groups
for name, group_df in grouped:
    print(f"Group: {name}")
    print(group_df)
    print()

In [ ]:
# Get a specific group
grouped.get_group('A')

In [ ]:
# Basic aggregation
df.groupby('group')['math'].mean()

In [ ]:
# Multiple columns
df.groupby('group')[['math', 'physics', 'chemistry']].mean()

### `agg` - Multiple Aggregations

The `agg` (or `aggregate`) method allows applying multiple aggregation functions at once, and even different functions to different columns.

In [ ]:
# Single aggregation (equivalent to .mean())
df.groupby('group')['math'].agg('mean')

In [ ]:
# Multiple aggregations on one column
df.groupby('group')['math'].agg(['mean', 'std', 'min', 'max'])

In [ ]:
# Multiple aggregations on multiple columns
df.groupby('group')[['math', 'physics']].agg(['mean', 'std'])

In [ ]:
# Different aggregations for different columns using a dictionary
df.groupby('group').agg({
    'math': 'mean',
    'physics': ['min', 'max'],
    'chemistry': 'std'
})

In [ ]:
# Named aggregations (cleaner column names)
df.groupby('group').agg(
    math_avg=('math', 'mean'),
    physics_range=('physics', lambda x: x.max() - x.min()),
    chemistry_std=('chemistry', 'std')
)

### `transform` with `groupby` - Broadcasting Group Results

The real power of `transform` shines when combined with `groupby`. It broadcasts group-level calculations back to each row, maintaining the original DataFrame's shape.

In [ ]:
# Problem: Add a column with each student's group average
# Using apply - returns aggregated result (3 rows)
df.groupby('group')['math'].apply(lambda x: x.mean())

In [ ]:
# Using transform - broadcasts back to original shape (6 rows)
df.groupby('group')['math'].transform('mean')

In [ ]:
# Adding group mean as a new column
df_with_group_mean = df.assign(
    group_math_avg=df.groupby('group')['math'].transform('mean')
)
df_with_group_mean

In [ ]:
# Normalize within each group (z-score per group)
df.assign(
    math_normalized=df.groupby('group')['math'].transform(
        lambda x: (x - x.mean()) / x.std()
    )
)

In [ ]:
# Rank within each group
df.assign(
    math_rank_in_group=df.groupby('group')['math'].transform('rank', ascending=False)
)

### `apply` with `groupby` - Flexible Group Operations

`apply` on a grouped object is the most flexible option. The function receives the entire group as a DataFrame and can return:
- A scalar (aggregation)
- A Series (transformation)
- A DataFrame (complex transformations)

In [ ]:
# Custom function that returns a scalar per group
def grade_spread(group):
    return group['math'].max() - group['math'].min()

df.groupby('group').apply(grade_spread, include_groups=False)

In [ ]:
# Custom function that returns a DataFrame (e.g., top student per group)
def top_student(group):
    return group.nlargest(1, 'math')

df.groupby('group').apply(top_student, include_groups=False)

### Summary: `agg` vs `transform` vs `apply` with groupby

| Method | Output Shape | Use Case |
|--------|--------------|----------|
| `agg` | One row per group | Summary statistics, multiple aggregations |
| `transform` | Same as input | Broadcasting group stats back to rows |
| `apply` | Flexible | Complex operations, custom logic |

## The `assign` Method

`assign` creates new columns (or overwrites existing ones) and returns a new DataFrame. It's particularly useful for method chaining.

In [ ]:
# Traditional way to add a column
df_copy = df.copy()
df_copy['average'] = (df_copy['math'] + df_copy['physics'] + df_copy['chemistry']) / 3
df_copy

In [ ]:
# Using assign (doesn't modify original)
df.assign(
    average=(df['math'] + df['physics'] + df['chemistry']) / 3
)

In [ ]:
# assign with lambda - references the DataFrame being transformed
df.assign(
    average=lambda x: (x['math'] + x['physics'] + x['chemistry']) / 3
)

In [ ]:
# Multiple columns at once, with dependencies between them
df.assign(
    average=lambda x: (x['math'] + x['physics'] + x['chemistry']) / 3,
    passed=lambda x: x['average'] >= 80,  # Uses the average column just created!
    group_avg=lambda x: x.groupby('group')['average'].transform('mean')
)

**Why use `assign`?**
- Supports method chaining (functional style)
- Doesn't modify the original DataFrame
- Lambda functions can reference columns created in the same `assign` call

In [ ]:
# Method chaining example
(
    df
    .assign(average=lambda x: x[['math', 'physics', 'chemistry']].mean(axis=1))
    .query('average >= 80')
    .sort_values('average', ascending=False)
)

## Reshaping Data: `melt` and `pivot`

### Wide vs Long Format

Data can be organized in two main formats:
- **Wide format**: Each variable has its own column (e.g., `math`, `physics`, `chemistry` as separate columns)
- **Long format**: Variables are stacked into two columns: one for the variable name, one for the value

Different analyses require different formats. `melt` converts wide to long, `pivot` converts long to wide.

In [ ]:
# Our data is currently in wide format
df

### `melt` - Wide to Long

`melt` "unpivots" a DataFrame from wide to long format.

In [ ]:
# Convert to long format
df_long = df.melt(
    id_vars=['student', 'group'],      # Columns to keep as identifiers
    value_vars=['math', 'physics', 'chemistry'],  # Columns to unpivot
    var_name='subject',                # Name for the variable column
    value_name='grade'                 # Name for the value column
)
df_long

In [ ]:
# If value_vars is omitted, all columns not in id_vars are melted
df.melt(id_vars=['student', 'group'], var_name='subject', value_name='grade')

**When is long format useful?**
- Aggregating across categories (e.g., average grade per subject)
- Many visualization libraries prefer long format
- Easier to filter by category

In [ ]:
# Average grade per subject (easier in long format)
df_long.groupby('subject')['grade'].mean()

In [ ]:
# Average grade per group and subject
df_long.groupby(['group', 'subject'])['grade'].mean()

### `pivot` and `pivot_table` - Long to Wide

`pivot` reshapes data from long to wide format. `pivot_table` is similar but handles duplicate entries by aggregating them.

In [ ]:
# pivot - convert back to wide format
df_wide = df_long.pivot(
    index=['student', 'group'],  # Columns to use as row identifiers
    columns='subject',            # Column whose values become new columns
    values='grade'                # Values to fill the new columns
)
df_wide

In [ ]:
# Clean up the result
df_wide.reset_index().rename_axis(None, axis=1)

**`pivot` fails with duplicate entries:**

In [ ]:
# If we try to pivot grouped data, it fails because there are multiple values per cell
try:
    df_long.pivot(index='group', columns='subject', values='grade')
except ValueError as e:
    print(f"Error: {e}")

### `pivot_table` - Handles Duplicates with Aggregation

In [ ]:
# pivot_table aggregates duplicates (default is mean)
df_long.pivot_table(
    index='group',
    columns='subject',
    values='grade',
    aggfunc='mean'
)

In [ ]:
# Multiple aggregations
df_long.pivot_table(
    index='group',
    columns='subject',
    values='grade',
    aggfunc=['mean', 'std']
)

In [ ]:
# Add margins (totals)
df_long.pivot_table(
    index='group',
    columns='subject',
    values='grade',
    aggfunc='mean',
    margins=True,
    margins_name='Overall'
)

### `pivot_table` vs `groupby` + `agg`

`pivot_table` is essentially a `groupby` followed by reshaping. These are equivalent:

In [ ]:
# Using pivot_table
pt = df_long.pivot_table(index='group', columns='subject', values='grade', aggfunc='mean')
pt

In [ ]:
# Equivalent using groupby
gb = df_long.groupby(['group', 'subject'])['grade'].mean().unstack()
gb

## Summary

| Operation | Purpose | Key Behavior |
|-----------|---------|-------------|
| `map` | Element-wise on Series | Dictionary/function mapping |
| `apply` | Flexible transformation | Works on Series, rows, or columns |
| `transform` | Same-shape transformation | Essential for group broadcasting |
| `groupby` | Split-apply-combine | Foundation for group operations |
| `agg` | Multiple aggregations | Reduce groups to summary values |
| `assign` | Add columns fluently | Method chaining, immutable |
| `melt` | Wide → Long | Unpivot columns to rows |
| `pivot` | Long → Wide (unique) | Reshape without aggregation |
| `pivot_table` | Long → Wide (duplicates) | Reshape with aggregation |